In [1]:
from anthropic import Anthropic
from dotenv import load_dotenv
from building_with_the_claude_api import add_assistant_message, add_user_message, chat, Effort

load_dotenv()
client = Anthropic()
system_prompt = """Behave as if you are explain to a history professor with no STEM background.
Their son wants to major in one of those fields."""

In [2]:
messages = []

while True:
    user_input = input("> ")
    print("---")
    print(user_input)

    add_user_message(messages=messages, text=user_input)
    response = chat(messages=messages, client=client, system_prompt=system_prompt, effort=Effort.LOW)
    add_assistant_message(messages=messages, text=response)
    print("---")
    print(response)


---
what is LLMOps

---
# LLMOps, Explained for a Historian

Think of it this way: your son might someday take a really talented but unpredictable graduate student — a Large Language Model (LLM) like ChatGPT — and try to put that student's talents to reliable, professional use in a company or product.

**LLMOps** (Large Language Model Operations) is the set of practices and tools for managing that "graduate student" responsibly at scale.

## An Analogy You'll Appreciate

Imagine you've discovered a brilliant research assistant who has read almost every book in the world, but:

- Sometimes confidently states false things (this is called "hallucinating")
- Can't remember previous conversations unless you remind them
- Gives different answers depending on subtle wording changes
- Is expensive to consult and slow if you ask too much at once
- Might occasionally say something inappropriate or biased

If you wanted to build a reliable research operation using dozens of copies of this assista

KeyboardInterrupt: Interrupted by user

In [3]:
# Streaming entire message events
messages = []
add_user_message(messages=messages, text="Generate one sentence ML eng portfolio project idea.")

request = {
    "model": "claude-sonnet-5",
    "max_tokens": 1024,
    "messages": messages,
    "output_config": {"effort": Effort.MEDIUM},
    "stream": True
}

stream = client.messages.create(**request)

for message in stream:
    print(message)

RawMessageStartEvent(message=Message(id='msg_011CephYrQCpFJeqUSxC2GqS', container=None, content=[], model='claude-sonnet-5', role='assistant', stop_details=None, stop_reason=None, stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=25, output_tokens=1, output_tokens_details=None, server_tool_use=None, service_tier='standard')), type='message_start')
RawContentBlockStartEvent(content_block=TextBlock(citations=None, text='', type='text'), index=0, type='content_block_start')
RawContentBlockDeltaEvent(delta=TextDelta(text='**', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text='Project Idea:** Build an end-to-end system that predicts customer churn for a subscription-based business by scraping/using', type='text_delta'), index=0, type='content_block_delta')
Ra

In [4]:
# Streaming text response
messages = []
add_user_message(messages=messages, text="Generate one sentence ML eng portfolio project idea.")

request = {
    "model": "claude-sonnet-5",
    "max_tokens": 1024,
    "messages": messages,
    "output_config": {"effort": Effort.MEDIUM},
}

with client.messages.stream(**request) as stream:
    for text in stream.text_stream:
        print(text, end="")

print("\n---")
stream.get_final_message()

**Project Idea:** Build an end-to-end ML pipeline that predicts customer churn for a subscription-based business by scraping/using synthetic transactional data, engineering behavioral features (usage frequency, support tickets, payment history), training and comparing multiple models (XGBoost, Random Forest, Neural Network) with hyperparameter tuning, then deploying the best model as a REST API with a Streamlit dashboard for real-time predictions and SHAP-based explainability, all containerized with Docker and monitored using MLflow for experiment tracking.
---


ParsedMessage(id='msg_011CephZQAuYwqyrL82yk78d', container=None, content=[ParsedTextBlock(citations=None, text='**Project Idea:** Build an end-to-end ML pipeline that predicts customer churn for a subscription-based business by scraping/using synthetic transactional data, engineering behavioral features (usage frequency, support tickets, payment history), training and comparing multiple models (XGBoost, Random Forest, Neural Network) with hyperparameter tuning, then deploying the best model as a REST API with a Streamlit dashboard for real-time predictions and SHAP-based explainability, all containerized with Docker and monitored using MLflow for experiment tracking.', type='text', parsed_output=None)], model='claude-sonnet-5', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inferen

In [5]:
# Structured data: Claude wrap the json with some sort of explaination
messages = []

add_user_message(messages=messages, text="Generate a very short event bridge rule as json")
text = chat(messages=messages, client=client)
text


'Here\'s a very short EventBridge rule example in JSON:\n\n```json\n{\n  "source": ["aws.ec2"],\n  "detail-type": ["EC2 Instance State-change Notification"],\n  "detail": {\n    "state": ["running"]\n  }\n}\n```\n\nThis rule triggers when an EC2 instance changes to the "running" state.'

In [6]:
# Structured data: force a JSON-only response via structured outputs
# (claude-sonnet-5 dropped assistant message prefill, so the classic
# "```json" prefill trick no longer works -- use output_config.format instead)
import json
messages = []

add_user_message(messages=messages, text="Generate a very short event bridge rule as json")

event_bridge_rule_schema = {
    "type": "object",
    "properties": {
        "source": {"type": "array", "items": {"type": "string"}},
        "detail-type": {"type": "array", "items": {"type": "string"}},
        "detail": {
            "type": "object",
            "properties": {"state": {"type": "array", "items": {"type": "string"}}},
            "required": ["state"],
            "additionalProperties": False,
        },
    },

    "required": ["source", "detail-type", "detail"],
    "additionalProperties": False,
}

text = chat(messages=messages, client=client, effort=Effort.LOW, json_schema=event_bridge_rule_schema)

json.loads(text)


{'source': ['aws.ec2'],
 'detail-type': ['EC2 Instance State-change Notification'],
 'detail': {'state': ['running']}}

In [7]:
# Structured data: generate N macOS CLI commands as JSON
messages = []

n = 5
commands_schema = {
    "type": "object",
    "properties": {
        "commands": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "command": {"type": "string"},
                    "description": {"type": "string"},
                },
                "required": ["command", "description"],
                "additionalProperties": False,
            },
        }
    },
    "required": ["commands"],
    "additionalProperties": False,
}

add_user_message(messages=messages, text=f"Generate exactly {n} macOS CLI commands for managing running processes")
text = chat(messages=messages, client=client, effort=Effort.LOW, json_schema=commands_schema)

result = json.loads(text)
print(result)

print("---")
for c in result["commands"]:
    print(c["command"], "-", c["description"])


{'commands': [{'command': 'ps aux', 'description': 'Display all running processes with detailed information including CPU and memory usage'}, {'command': 'top -o cpu', 'description': 'Show real-time list of processes sorted by CPU usage'}, {'command': 'kill -9 <PID>', 'description': 'Forcefully terminate a process using its process ID'}, {'command': 'pkill -f <process_name>', 'description': 'Terminate all processes matching a given name pattern'}, {'command': 'lsof -i :<port>', 'description': 'List processes that are using a specific network port'}]}
---
ps aux - Display all running processes with detailed information including CPU and memory usage
top -o cpu - Show real-time list of processes sorted by CPU usage
kill -9 <PID> - Forcefully terminate a process using its process ID
pkill -f <process_name> - Terminate all processes matching a given name pattern
lsof -i :<port> - List processes that are using a specific network port
